# Feature Space Analysis: alpha_feat vs alpha_topo

This notebook generates SynCo graphs with fixed structural hyperparameters and varies only `alpha_feat` and `alpha_topo`. It projects node attributes with t-SNE and saves a publication-style figure in `experiments/feature_space/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if ROOT.name in {"experiments", "examples"}:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from synco import SynCoGenerator

OUTPUT_DIR = ROOT / "experiments" / "feature_space"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

## Experiment Setup

The graph structure is kept fixed. Each panel uses the same seed and structural parameters, changing only the feature noise strength and topology smoothing weight.

In [ ]:
SEED = 2026
N = 900
Y = [300, 300, 300]
E = [900, 1_200, 900]
RHO = 4_000
FEATURE_DIM = 60

ALPHA_FEAT_VALUES = [0.05, 0.30, 0.80]
ALPHA_TOPO_VALUES = [0.00, 0.50, 0.90]

BASE_PARAMS = {
    "n": N,
    "y": Y,
    "k": len(Y),
    "e": E,
    "A_in": [
        torch.tensor([[1.0]]),
        torch.tensor([[0.6, 0.3, 0.1], [0.1, 0.7, 0.2], [0.0, 0.3, 0.7]]),
        torch.tensor([[0.9, 0.1], [0.1, 0.9]]),
    ],
    "dst": ["power_law", "normal", "uniform"],
    "rho": RHO,
    "A_out": torch.tensor([[0.0, 0.2, 0.8], [0.3, 0.0, 0.7], [0.5, 0.5, 0.0]]),
    "S": [torch.tensor([1.0]), torch.tensor([0.4, 0.3, 0.3]), torch.tensor([0.9, 0.1])],
    "d": FEATURE_DIM,
    "build_subgraphs": False,
    "connect_isolated_nodes": False,
}

pd.DataFrame([{
    "n": N,
    "rho": RHO,
    "classes": Y,
    "intraclass_edges": E,
    "alpha_feat_values": ALPHA_FEAT_VALUES,
    "alpha_topo_values": ALPHA_TOPO_VALUES,
}])

## Generate Feature Matrices

In [ ]:
feature_frames = []
summary_records = []

for alpha_topo in ALPHA_TOPO_VALUES:
    for alpha_feat in ALPHA_FEAT_VALUES:
        params = {
            **BASE_PARAMS,
            "alpha_feat": alpha_feat,
            "alpha_topo": alpha_topo,
        }
        graph = SynCoGenerator(seed=SEED).generate(**params)
        x = graph.x.detach().cpu().numpy().astype(np.float64)
        y = graph.y.detach().cpu().numpy().astype(int)

        frame = pd.DataFrame(x, columns=[f"x_{idx}" for idx in range(x.shape[1])])
        frame["class"] = y
        frame["node"] = np.arange(len(y))
        frame["alpha_feat"] = alpha_feat
        frame["alpha_topo"] = alpha_topo
        feature_frames.append(frame)

        summary_records.append({
            "alpha_feat": alpha_feat,
            "alpha_topo": alpha_topo,
            "n_actual": graph.num_nodes(),
            "edges_actual": graph.num_edges(),
        })
        print(f"alpha_feat={alpha_feat:.2f} alpha_topo={alpha_topo:.2f} edges={graph.num_edges()}")

features = pd.concat(feature_frames, ignore_index=True)
summary = pd.DataFrame(summary_records)
summary.to_csv(OUTPUT_DIR / "feature_space_alpha_summary.csv", index=False)
features.shape, summary

## t-SNE Projection

The t-SNE is fit once over all generated feature matrices, after standardization and PCA pre-reduction. This keeps all panels in a shared projection space.

In [ ]:
feature_columns = [column for column in features.columns if column.startswith("x_")]
x_scaled = StandardScaler().fit_transform(features[feature_columns])
x_pca = PCA(n_components=min(30, len(feature_columns)), random_state=SEED).fit_transform(x_scaled)

try:
    embedding = TSNE(
        n_components=2,
        perplexity=35,
        init="pca",
        learning_rate=200,
        max_iter=1_000,
        random_state=SEED,
    ).fit_transform(x_pca)
except TypeError:
    embedding = TSNE(
        n_components=2,
        perplexity=35,
        init="pca",
        learning_rate=200,
        n_iter=1_000,
        random_state=SEED,
    ).fit_transform(x_pca)

features["tsne_1"] = embedding[:, 0]
features["tsne_2"] = embedding[:, 1]
features.to_csv(OUTPUT_DIR / "feature_space_alpha_tsne.csv", index=False)
features[["node", "class", "alpha_feat", "alpha_topo", "tsne_1", "tsne_2"]].head()

## TikZ Publication Figure

In [ ]:
MAX_POINTS_PER_CLASS = 250
PLOT_ALPHA_TOPO_VALUES = [ALPHA_TOPO_VALUES[0], ALPHA_TOPO_VALUES[2]]

tikz_path = OUTPUT_DIR / "feature_space_alpha_tsne_tikz.tex"
tikz_points_path = OUTPUT_DIR / "feature_space_alpha_tsne_tikz_points.csv"

marker_by_class = {
    0: "o",
    1: "square",
    2: "triangle",
}

plot_frames = []
panel_blocks = []

for alpha_topo in PLOT_ALPHA_TOPO_VALUES:
    for alpha_feat in ALPHA_FEAT_VALUES:
        panel = features[
            np.isclose(features["alpha_topo"], alpha_topo)
            & np.isclose(features["alpha_feat"], alpha_feat)
        ]
        axis_lines = [
            r"\nextgroupplot[",
            rf"    xlabel={{$\alpha_{{feat}}={alpha_feat}$, $\alpha_{{topo}}={alpha_topo}$}},",
            r"    xtick=\empty,",
            r"    ytick=\empty,",
            r"]",
        ]

        for class_id, marker in marker_by_class.items():
            points = panel[panel["class"] == class_id]
            if len(points) > MAX_POINTS_PER_CLASS:
                points = points.sample(MAX_POINTS_PER_CLASS, random_state=SEED)
            plot_frames.append(points)

            coordinates = "\n".join(
                f"        ({row.tsne_1:.6f},{row.tsne_2:.6f})"
                for row in points.itertuples()
            )
            axis_lines.extend([
                (
                    rf"\addplot+[only marks, mark={marker}, mark size=1.0pt, black, "
                    r"mark options={solid, fill=white, line width=0.25pt}] coordinates {"
                ),
                coordinates,
                r"};",
            ])

        panel_blocks.append("\n".join(axis_lines))

plot_features = pd.concat(plot_frames, ignore_index=True)
plot_features = plot_features[["tsne_1", "tsne_2", "class", "alpha_feat", "alpha_topo"]]
plot_features.to_csv(tikz_points_path, index=False)

tikz_header = r"""
\documentclass[tikz,border=4pt]{standalone}
\usepackage{pgfplots}
\usepgfplotslibrary{groupplots}
\usetikzlibrary{plotmarks}
\pgfplotsset{compat=1.18}

\begin{document}
\begin{tikzpicture}
\begin{groupplot}[
    group style={group size=3 by 2, horizontal sep=0.55cm, vertical sep=0.95cm},
    width=4.05cm,
    height=3.35cm,
    scale only axis,
    axis line style={black, line width=0.35pt},
    tick style={black, line width=0.35pt},
    xlabel style={font=\footnotesize, yshift=0.45ex},
    ylabel style={font=\footnotesize},
    clip marker paths=true,
]
""".strip()

tikz_footer = r"""
\end{groupplot}

\node[anchor=south, font=\footnotesize] at ([yshift=0.18cm]current bounding box.north) {
\begin{tabular}{ccc}
\tikz[baseline=-0.55ex]{\draw plot[only marks, mark=o, mark size=1.3pt, mark options={solid, fill=white, line width=0.35pt}] coordinates {(0,0)};} Classe 0 &
\tikz[baseline=-0.55ex]{\draw plot[only marks, mark=square, mark size=1.3pt, mark options={solid, fill=white, line width=0.35pt}] coordinates {(0,0)};} Classe 1 &
\tikz[baseline=-0.55ex]{\draw plot[only marks, mark=triangle, mark size=1.5pt, mark options={solid, fill=white, line width=0.35pt}] coordinates {(0,0)};} Classe 2
\end{tabular}
};

\end{tikzpicture}
\end{document}
""".strip()

panel_source = "\n\n".join(panel_blocks)
tikz_source = f"{tikz_header}\n\n{panel_source}\n\n{tikz_footer}\n"
tikz_path.write_text(tikz_source)

print(f"TikZ figure: {tikz_path}")
print(f"Plotted points: {tikz_points_path}")